In [24]:
import pandas as pd

In [25]:
census_years = [1801,
                1850,
                1860,
                1880,
                1901]

filelist = ['data/census_1801__v1__cl.csv.gz',
            'data/census_1850__v1__cl.csv.gz',
            'data/census_1860__v1__cl.csv.gz',
            'data/census_1880__v1__cl.csv.gz',
            'data/census_1901__v1__cl.csv.gz']

In [26]:
census_df_list = []

for i in range(len(census_years)):
    census_df_year = pd.read_csv(filelist[i], usecols=['handicaps'])
    census_df_year['census_year'] = census_years[i]
    census_df_list.append(census_df_year)

census_df = pd.concat(census_df_list, ignore_index=True)

/tmp/ipykernel_13871/20197055.py:4: DtypeWarning: Columns (16: handicaps) have mixed types. Specify dtype option on import or set low_memory=False.
  census_df_year = pd.read_csv(filelist[i], usecols=['handicaps'])


In [27]:
census_df.dropna().info()

<class 'pandas.DataFrame'>
Index: 144468 entries, 815170 to 8538607
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   handicaps    144468 non-null  str  
 1   census_year  144468 non-null  int64
dtypes: int64(1), str(1)
memory usage: 3.3 MB


In [28]:
row_counts = census_df.groupby('census_year').size()

print('Row counts per census year:')
print(row_counts)

Row counts per census year:
census_year
1801     937944
1850    1405217
1860    1748226
1880    1979455
1901    2468040
dtype: int64


In [29]:
census_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8538882 entries, 0 to 8538881
Data columns (total 2 columns):
 #   Column       Dtype
---  ------       -----
 0   handicaps    str  
 1   census_year  int64
dtypes: int64(1), str(1)
memory usage: 130.3 MB


In [30]:
distinct_counts = census_df.groupby('census_year').nunique()

print('Distinct values per census year:')
print(distinct_counts)

distinct_counts.to_csv('output/distinct_counts.csv')

Distinct values per census year:
             handicaps
census_year           
1801                 2
1850              1832
1860              1301
1880               856
1901               881


In [31]:
census_df = census_df.dropna()

census_df = census_df.drop_duplicates()

census_df.to_csv('output/handicaps_distinct.csv')

In [32]:
word_counts = (
    census_df['handicaps']
    .str.lower()
    .str.findall(r'[a-zæøå]+')
    .explode()
    .value_counts()
)

word_counts.to_csv('output/word_counts.csv')

In [33]:
labels = {
    'forstand berøvet': ['forstand'],
    'blind':            ['blind'],
    'døvstum':          ['døvstum'],
    'døv':              ['døv']
}

for label, words in labels.items():
    for word in words:
        census_df.loc[census_df['handicaps'].str.contains(word,  case=False, na=False), 'handicaps_std'] = label

In [34]:
len(census_df) - len(census_df.dropna())

2304